### AwesomeAgent

This notebook demonstrates how to configure, deploy, and invoke an AWS BedrockAgentCore agent using the (now old) agentcore CLI.
Steps covered:
- AWS authentication setup
- Agent configuration
- Agent deployment
- Agent invocation

#### 0. Basic set-up of AWS credentials and `agentcore` cli availability.

##### 0.1 Make sure that we have valid AWS credentials.

Prepare environment variables

In [ ]:
export AWS_ACCOUNT_ID=832391729540
export AWS_REGION=eu-west-1
export AWS_LOGIN_PROFILE=agentcore
export AWS_AGENTCORE_ROLE_NAME=agentcore-starter-toolkit

In [ ]:
# login into AWS IAM account that has permissions to assume the agentcore-enabled role
aws login --profile $AWS_LOGIN_PROFILE --region $AWS_REGION

# assume the agentcore-enabled role to get temporary credentials
aws sts assume-role \
  --role-arn arn:aws:iam::$AWS_ACCOUNT_ID:role/$AWS_AGENTCORE_ROLE_NAME \
  --profile $AWS_LOGIN_PROFILE \
  --role-session-name agentcore-starter-toolkit \
  --output off

# export the temporary credentials as environment variables
eval $(aws configure export-credentials --profile $AWS_LOGIN_PROFILE --format env)

# verify that the temporary credentials are set correctly
aws configure list

##### 0.2 Make sure that we `agentcore` cli is available

In [ ]:
export VIRTUAL_ENV_DISABLE_PROMPT=1
source .venv/bin/activate
which agentcore

#### 1. Configure the agent with our existing code, using [`runtime.ts`](./src/runtime.ts) as entrypoint.

In [ ]:
agentcore configure \
    --entrypoint ./src/runtime.ts \
    --disable-memory \
    --name AwesomeAgent \
    --region $AWS_REGION \
    --non-interactive

This generates the agent configuration as [`.bedrock_agentcore.yaml`](.bedrock_agentcore.yaml) and the [`Dockerfile`](.bedrock_agentcore/AwesomeAgent/Dockerfile).

---

#### 2. Deploy configured agent to **Bedrock AgentCore**.

In [ ]:
agentcore deploy --auto-update-on-conflict

This builds the project and creates all the necessary AWS resources: Bedrock AgentCore Runtime, CodeBuild project, ECR, execution roles, log groups, etc.

---

#### 3. Invoke the new agent on Bedrock AgentCore platform.

Use as payload the schema defined by [`runtime.ts`](./src/runtime.ts).

In [ ]:
agentcore invoke '{"prompt": "I am fairly awesome now, but want to be less awesome, just like bedrock-agentcore-starter-toolkit"}'

---

#### 4. Destroy the agent and cleanup AWS resources.

In [ ]:
agentcore destroy --force